<a href="https://colab.research.google.com/github/2420039806-a11y/Compiler-designing-programs/blob/main/LEX_and_YACC_Compiler.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LEX and YACC Compiler in Colab

Drawbacks:
* Regular interrupts (Ctrl+D, Ctrl+C) for shell won't work in Colab while inputting for program.
<br>Workaround: Store your inputs in a txt file and pass it to the program.

In [ ]:
#@title Install *prerqeuisites* (run this cell first to work on LEX/YACC)
!sudo apt install flex bison

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
flex is already the newest version (2.6.4-8.2build1).
bison is already the newest version (2:3.8.2+dfsg-1build2).
0 upgraded, 0 newly installed, 0 to remove and 0 not upgraded.


## Lex only

### **1) Write a LEX program to count the number of words, characters, spaces, and lines in a given input file**

In [ ]:
#@title Writing Lex program
%%writefile program.l

%{
    #include <stdio.h>
    int ctChar=0;
    int ctSpace=0;
    int ctWord=0;
    int ctLine=0;
%}
WORD [^ \t\n,\.:]+
EOL [\n]
BLANK [ ]
%%

{WORD} {ctWord++; ctChar+=yyleng;}
{BLANK} {ctSpace++;}
{EOL} {ctLine++;}
. {ctChar++;}
%%

void main(int argc, char *argv[]){
    if(argc!=2){
        printf("Usage:\n\t./a.out <FILENAME>\n");
        exit(0);
    }

    yyin=fopen(argv[1],"r");
    yylex();

    printf("Word Count: %d\n",ctWord);
    printf("Character Count: %d\n",ctChar);
    printf("Space Count: %d\n",ctSpace);
    printf("Line Count: %d\n",ctLine);
    fclose(yyin);

}

int yywrap(){
    return 1;
}

Overwriting program.l


In [ ]:
#@title Shell Execution (you can rewrite the commands as per your need, eg. if you want to include a file as an input)
%%shell

lex -l program.l
gcc lex.yy.c
./a.out program.txt

Word Count: 5
Character Count: 18
Space Count: 4
Line Count: 2


## **3) Write a LEX and YACC program to recognize/validate a valid identifier.**

In [ ]:
#@title Writing YACC program
%%writefile program.y

%{
    #include<stdio.h>
    #include<stdlib.h>
%}
%token DIGIT LETTER UND NL
%%
stmt: variable NL {printf("Valid Identifier\n");exit(0);}
variable: LETTER alphanumeric;
alphanumeric: LETTER alphanumeric
            | DIGIT alphanumeric
            | UND alphanumeric
            | LETTER
            | DIGIT
            | UND;
%%

int yyerror(){
    printf("Invalid Identifier\n");
    exit(0);
}

void main(){
    printf("Enter the variable name: ");
    yyparse();
}

Overwriting program.y


# **4) Write a LEX program to identify and return tokens for letters, digits, underscores, and newline characters.**

In [ ]:
#@title Writing Lex program
%%writefile program.l

%{
    #include "y.tab.h"
%}
%%
[a-zA-Z] {return LETTER;}
[0-9] {return DIGIT;}
[_] {return UND;}
\n {return NL;}
. {return yytext[0];}
%%

Overwriting program.l


## **5) Write a LEX and YACC program to validate whether a given string is a valid identifier, and execute the program using YACC and LEX.**

In [ ]:
#@title Shell Execution (you can rewrite the commands as per your need, eg. if you want to include a file as an input)
%%shell

yacc -d program.y
lex program.l
cc y.tab.c lex.yy.c -ll
./a.out

y.tab.c: In function ‘yyparse’:
y.tab.c:1121:16: warning: implicit declaration of function ‘yylex’ [-Wimplicit-function-declaration]
       yychar = yylex ();
                ^~~~~
y.tab.c:1256:7: warning: implicit declaration of function ‘yyerror’; did you mean ‘yyerrok’? [-Wimplicit-function-declaration]
       yyerror (YY_("syntax error"));
       ^~~~~~~
       yyerrok
Enter the variable name: variable_name
Valid Identifier


# **7) Write a C program to eliminate direct left recursion from a given context-free grammar production rule.**

In [ ]:
%%writefile left_recursion.c

#include <stdio.h>
#include <string.h>

int main() {
    char rule[100];
    char lhs;
    char alpha[50], beta[50];

    printf("Enter production rule (e.g., A->Aa|b): ");
    scanf("%99s", rule);

    // Get the left-hand side non-terminal
    lhs = rule[0];

    // Check for -> symbol
    if (rule[1] != '-' || rule[2] != '>') {
        printf("\nInvalid production rule format.\n");
        return 1;
    }

    // Check whether the rule has direct left recursion
    if (rule[3] != lhs) {
        printf("\nNo direct left recursion detected.\n");
        printf("Original Rule: %s\n", rule);
        return 0;
    }

    // Find '|'
    char *pipe = strchr(rule, '|');

    if (pipe == NULL) {
        printf("\nError: Production must contain '|'.\n");
        return 1;
    }

    // Extract alpha: characters after A->A
    int alphaLength = pipe - (rule + 4);

    strncpy(alpha, rule + 4, alphaLength);
    alpha[alphaLength] = '\0';

    // Extract beta: characters after '|'
    strcpy(beta, pipe + 1);

    printf("\nOriginal Rule:\n");
    printf("%s\n", rule);

    printf("\nAfter Eliminating Left Recursion:\n");

    printf("%c -> %s%c'\n", lhs, beta, lhs);
    printf("%c' -> %s%c' | ε\n", lhs, alpha, lhs);

    return 0;
}

Overwriting left_recursion.c


In [ ]:
%%shell
gcc left_recursion.c -o left_recursion
echo "A->Aa|b" | ./left_recursion

Enter production rule (e.g., A->Aa|b): 
Original Rule:
A->Aa|b

After Eliminating Left Recursion:
A -> bA'
A' -> aA' | ε


## **8) Implement Program for Constructing LL(1) Parsing table**

In [ ]:

%%writefile ll1.c
#include <stdio.h>
#include <string.h>

#define MAX 30
#define LEN 100
#define ASCII 128

char prod[MAX][LEN];
char first[26][ASCII], follow[26][ASCII];
char table[26][ASCII][LEN];

int n;
int nt[26], terminal[ASCII];

int isNT(char c) {
    return c >= 'A' && c <= 'Z';
}

int add(char set[], char c) {
    if (c == '\0' || strchr(set, c))
        return 0;

    int len = strlen(set);
    set[len] = c;
    set[len + 1] = '\0';
    return 1;
}

/* Find FIRST of a string */
void firstString(const char *s, char result[]) {
    result[0] = '\0';

    if (s[0] == '\0' || s[0] == '#') {
        add(result, '#');
        return;
    }

    int nullable = 1;

    for (int i = 0; s[i]; i++) {
        char c = s[i];

        if (!isNT(c)) {
            add(result, c);
            nullable = 0;
            break;
        }

        int index = c - 'A';

        for (int j = 0; first[index][j]; j++) {
            if (first[index][j] != '#')
                add(result, first[index][j]);
        }

        if (!strchr(first[index], '#')) {
            nullable = 0;
            break;
        }
    }

    if (nullable)
        add(result, '#');
}

/* Calculate FIRST sets */
void calculateFirst() {
    int changed;

    do {
        changed = 0;

        for (int i = 0; i < n; i++) {
            int A = prod[i][0] - 'A';
            char *rhs = prod[i] + 3;

            if (strcmp(rhs, "#") == 0) {
                changed |= add(first[A], '#');
                continue;
            }

            int nullable = 1;

            for (int j = 0; rhs[j]; j++) {
                char c = rhs[j];

                if (!isNT(c)) {
                    changed |= add(first[A], c);
                    nullable = 0;
                    break;
                }

                int B = c - 'A';

                for (int k = 0; first[B][k]; k++) {
                    if (first[B][k] != '#')
                        changed |= add(first[A],
                                       first[B][k]);
                }

                if (!strchr(first[B], '#')) {
                    nullable = 0;
                    break;
                }
            }

            if (nullable)
                changed |= add(first[A], '#');
        }

    } while (changed);
}

/* Calculate FOLLOW sets */
void calculateFollow() {
    int changed;

    add(follow[prod[0][0] - 'A'], '$');

    do {
        changed = 0;

        for (int i = 0; i < n; i++) {
            int A = prod[i][0] - 'A';
            char *rhs = prod[i] + 3;

            for (int j = 0; rhs[j]; j++) {
                if (!isNT(rhs[j]))
                    continue;

                int B = rhs[j] - 'A';
                char temp[ASCII];

                firstString(rhs + j + 1, temp);

                for (int k = 0; temp[k]; k++) {
                    if (temp[k] != '#')
                        changed |= add(follow[B],
                                       temp[k]);
                }

                if (strchr(temp, '#')) {
                    for (int k = 0; follow[A][k]; k++) {
                        changed |= add(follow[B],
                                       follow[A][k]);
                    }
                }
            }
        }

    } while (changed);
}

/* Print a set */
void printSet(char set[]) {
    printf("{ ");

    for (int i = 0; set[i]; i++) {
        printf("%c", set[i]);

        if (set[i + 1])
            printf(", ");
    }

    printf(" }");
}

/* Insert production into parsing table */
int insertTable(int A, char terminalSymbol,
                const char *production) {

    int t = (unsigned char)terminalSymbol;

    if (table[A][t][0] == '\0') {
        strcpy(table[A][t], production);
        return 0;
    }

    if (strcmp(table[A][t], production) != 0) {
        printf("\nConflict at M[%c,%c]: %s and %s\n",
               A + 'A', terminalSymbol,
               table[A][t], production);
        return 1;
    }

    return 0;
}

/* Construct LL(1) parsing table */
void constructTable() {
    int conflict = 0;

    for (int i = 0; i < n; i++) {
        int A = prod[i][0] - 'A';
        char *rhs = prod[i] + 3;

        char temp[ASCII];

        firstString(rhs, temp);

        /* Rule 1: Use FIRST of RHS */
        for (int j = 0; temp[j]; j++) {
            if (temp[j] != '#') {
                conflict |= insertTable(
                    A, temp[j], prod[i]
                );
            }
        }

        /* Rule 2: If epsilon exists, use FOLLOW */
        if (strchr(temp, '#')) {
            for (int j = 0; follow[A][j]; j++) {
                conflict |= insertTable(
                    A, follow[A][j], prod[i]
                );
            }
        }
    }

    printf("\n========== LL(1) PARSING TABLE ==========\n");

    printf("%-12s", "NT");

    for (int t = 0; t < ASCII; t++) {
        if (terminal[t])
            printf("%-15c", t);
    }

    printf("%-15s\n", "$");

    for (int A = 0; A < 26; A++) {
        if (!nt[A])
            continue;

        printf("%-12c", A + 'A');

        for (int t = 0; t < ASCII; t++) {
            if (terminal[t]) {
                if (table[A][t][0])
                    printf("%-15s", table[A][t]);
                else
                    printf("%-15s", "-");
            }
        }

        if (table[A]['$'][0])
            printf("%-15s", table[A]['$']);
        else
            printf("%-15s", "-");

        printf("\n");
    }

    if (conflict)
        printf("\nGrammar is NOT LL(1).\n");
    else
        printf("\nGrammar is LL(1).\n");
}

int main() {
    printf("LL(1) PARSING TABLE CONSTRUCTION\n");

    printf("\nEnter number of productions: ");
    if (scanf("%d", &n) != 1 || n < 1 || n > MAX) {
        printf("Invalid number of productions.\n");
        return 1;
    }

    printf("\nEnter productions (example: S->AB)\n");
    printf("Use # for epsilon.\n\n");

    for (int i = 0; i < n; i++) {
        printf("Production %d: ", i + 1);

        if (scanf("%99s", prod[i]) != 1)
            return 1;

        if (!isNT(prod[i][0]) ||
            prod[i][1] != '-' ||
            prod[i][2] != '>' ||
            prod[i][3] == '\0') {

            printf("Invalid production format.\n");
            return 1;
        }

        nt[prod[i][0] - 'A'] = 1;

        char *rhs = prod[i] + 3;

        for (int j = 0; rhs[j]; j++) {
            if (rhs[j] == '#' &&
                !(rhs[0] == '#' && rhs[1] == '\0')) {
                printf("Use # only as a complete RHS.\n");
                return 1;
            }

            if (isNT(rhs[j])) {
                nt[rhs[j] - 'A'] = 1;
            }
            else if (rhs[j] != '#') {
                terminal[(unsigned char)rhs[j]] = 1;
            }
        }
    }

    calculateFirst();
    calculateFollow();

    printf("\n========== FIRST SETS ==========\n");

    for (int i = 0; i < 26; i++) {
        if (nt[i]) {
            printf("FIRST(%c) = ", i + 'A');
            printSet(first[i]);
            printf("\n");
        }
    }

    printf("\n========== FOLLOW SETS ==========\n");

    for (int i = 0; i < 26; i++) {
        if (nt[i]) {
            printf("FOLLOW(%c) = ", i + 'A');
            printSet(follow[i]);
            printf("\n");
        }
    }

    constructTable();

    return 0;
}

Writing ll1.c


In [ ]:

import subprocess

# Compile the C program
compile_result = subprocess.run(
    ["gcc", "ll1.c", "-o", "ll1"],
    capture_output=True,
    text=True
)

if compile_result.returncode != 0:
    print("Compilation error:")
    print(compile_result.stderr)
else:
    # Get input from the user
    n = int(input("Enter number of productions: "))

    productions = []

    for i in range(n):
        p = input(f"Production {i+1}: ")
        productions.append(p)

    # Prepare input for the C program
    data = str(n) + "\n"
    data += "\n".join(productions) + "\n"

    # Execute the C program
    result = subprocess.run(
        ["./ll1"],
        input=data,
        capture_output=True,
        text=True
    )

    print("\n" + result.stdout)

    if result.stderr:
        print(result.stderr)

Enter number of productions: 4
Production 1: S->AB
Production 2: A->a
Production 3: A->#
Production 4: B->b

LL(1) PARSING TABLE CONSTRUCTION

Enter number of productions: 
Enter productions (example: S->AB)
Use # for epsilon.

Production 1: Production 2: Production 3: Production 4: 
========== FIRST SETS ==========
FIRST(A) = { a, # }
FIRST(B) = { b }
FIRST(S) = { a, b }

========== FOLLOW SETS ==========
FOLLOW(A) = { b }
FOLLOW(B) = { $ }
FOLLOW(S) = { $ }

========== LL(1) PARSING TABLE ==========
NT          a              b              $              
A           A->a           A->#           -              
B           -              B->b           -              
S           S->AB          S->AB          -              

Grammar is LL(1).



# **8: Implement Program for Shift-Reduce parser**


In [ ]:
%%writefile shift_reduce.c

#include <stdio.h>
#include <string.h>

#define MAX 100
#define MAX_PROD 20

char production[MAX_PROD][MAX];
char stack[MAX];
int top = -1;
int n;

/* Display stack */
void displayStack()
{
    for (int i = 0; i <= top; i++)
        printf("%c", stack[i]);
}

/* Check whether the stack ends with a RHS
   and reduce it to the LHS */
int reduce()
{
    for (int p = 0; p < n; p++)
    {
        char lhs = production[p][0];

        /* Find -> */
        char *arrow = strstr(production[p], "->");

        if (arrow == NULL)
            continue;

        char *rhs = arrow + 2;
        int rhsLen = strlen(rhs);

        if (rhsLen == 0)
            continue;

        /* Check whether stack ends with RHS */
        if (top + 1 >= rhsLen)
        {
            int match = 1;

            for (int j = 0; j < rhsLen; j++)
            {
                if (stack[top - rhsLen + 1 + j] != rhs[j])
                {
                    match = 0;
                    break;
                }
            }

            if (match)
            {
                /* Remove RHS */
                top = top - rhsLen + 1;

                /* Push LHS */
                stack[top] = lhs;

                printf("Reduce %-20s Stack: ", production[p]);
                displayStack();
                printf("\n");

                return 1;
            }
        }
    }

    return 0;
}

int main()
{
    char input[MAX];

    printf("========================================\n");
    printf("        SHIFT-REDUCE PARSER\n");
    printf("========================================\n");

    /* Number of productions */
    printf("\nEnter number of productions: ");
    scanf("%d", &n);

    if (n <= 0 || n > MAX_PROD)
    {
        printf("Invalid number of productions.\n");
        return 1;
    }

    /* Read grammar */
    printf("\nEnter productions without spaces.\n");
    printf("Example: E->E+T\n\n");

    for (int i = 0; i < n; i++)
    {
        printf("Production %d: ", i + 1);
        scanf("%99s", production[i]);
    }

    /* Read input */
    printf("\nEnter input string: ");
    scanf("%99s", input);

    printf("\n========================================\n");
    printf("Grammar:\n");

    for (int i = 0; i < n; i++)
        printf("%s\n", production[i]);

    printf("Input: %s\n", input);

    printf("\n========================================\n");
    printf("SHIFT-REDUCE PROCESS\n");
    printf("========================================\n");

    int ip = 0;

    while (ip < strlen(input))
    {
        /* SHIFT */
        stack[++top] = input[ip];

        printf("Shift  '%c'             Stack: ",
               input[ip]);

        displayStack();
        printf("\n");

        ip++;

        /* REDUCE as much as possible */
        while (reduce())
        {
            /* Continue reducing */
        }
    }

    /* Final reductions */
    while (reduce())
    {
        /* Continue */
    }

    printf("\n========================================\n");

    /*
       Accept if stack contains only
       the start symbol.
    */
    if (top == 0 &&
        stack[0] == production[0][0])
    {
        printf("Result: ACCEPTED\n");
    }
    else
    {
        printf("Result: REJECTED\n");
    }

    printf("========================================\n");

    return 0;
}

Overwriting shift_reduce.c


In [ ]:
%%shell
gcc shift_reduce.c -o shift_reduce
./shift_reduce

        SHIFT-REDUCE PARSER

Enter number of productions: 3

Enter productions without spaces.
Example: E->E+T

Production 1:  E->E+E
Production 2: E->E*E
Production 3: E->i

Enter input string: i+i

Grammar:
E->E+E
E->E*E
E->i
Input: i+i

SHIFT-REDUCE PROCESS
Shift  'i'             Stack: i
Reduce E->i                 Stack: E
Shift  '+'             Stack: E+
Shift  'i'             Stack: E+i
Reduce E->i                 Stack: E+E
Reduce E->E+E               Stack: E

Result: ACCEPTED


## **10) Implementation of LR parser**

In [ ]:
%%writefile lr_parser.c

#include <stdio.h>
#include <string.h>

#define MAX_PROD 20
#define MAX_LEN 50
#define MAX_STATES 50
#define MAX_STACK 100

typedef struct {
    char lhs;
    char rhs[MAX_LEN];
} Production;

Production prod[MAX_PROD];
int n;

/* -------------------------------------------------- */
/* LR(0) Item                                          */
/* Example: A -> a.B                                  */
/* dot = position of dot in RHS                       */
/* -------------------------------------------------- */

typedef struct {
    int production;
    int dot;
} Item;

typedef struct {
    Item items[100];
    int count;
} State;

State states[MAX_STATES];
int stateCount = 0;

/* -------------------------------------------------- */
/* Check whether two items are equal                  */
/* -------------------------------------------------- */

int sameItem(Item a, Item b)
{
    return a.production == b.production &&
           a.dot == b.dot;
}

/* -------------------------------------------------- */
/* Add item if it doesn't already exist               */
/* -------------------------------------------------- */

int addItem(State *s, int p, int dot)
{
    Item newItem;
    newItem.production = p;
    newItem.dot = dot;

    for (int i = 0; i < s->count; i++)
    {
        if (sameItem(s->items[i], newItem))
            return 0;
    }

    s->items[s->count++] = newItem;
    return 1;
}

/* -------------------------------------------------- */
/* Closure operation                                   */
/* -------------------------------------------------- */

void closure(State *s)
{
    int changed = 1;

    while (changed)
    {
        changed = 0;

        for (int i = 0; i < s->count; i++)
        {
            int p = s->items[i].production;
            int dot = s->items[i].dot;

            if (dot >= strlen(prod[p].rhs))
                continue;

            char symbol = prod[p].rhs[dot];

            /* If symbol is a non-terminal */
            if (symbol >= 'A' && symbol <= 'Z')
            {
                for (int j = 0; j < n; j++)
                {
                    if (prod[j].lhs == symbol)
                    {
                        if (addItem(s, j, 0))
                            changed = 1;
                    }
                }
            }
        }
    }
}

/* -------------------------------------------------- */
/* GOTO operation                                      */
/* -------------------------------------------------- */

void goTo(State *from, char symbol, State *to)
{
    to->count = 0;

    for (int i = 0; i < from->count; i++)
    {
        int p = from->items[i].production;
        int dot = from->items[i].dot;

        if (dot < strlen(prod[p].rhs) &&
            prod[p].rhs[dot] == symbol)
        {
            addItem(to, p, dot + 1);
        }
    }

    closure(to);
}

/* -------------------------------------------------- */
/* Compare states                                      */
/* -------------------------------------------------- */

int sameState(State *a, State *b)
{
    if (a->count != b->count)
        return 0;

    for (int i = 0; i < a->count; i++)
    {
        int found = 0;

        for (int j = 0; j < b->count; j++)
        {
            if (sameItem(a->items[i], b->items[j]))
            {
                found = 1;
                break;
            }
        }

        if (!found)
            return 0;
    }

    return 1;
}

/* -------------------------------------------------- */
/* Find state                                          */
/* -------------------------------------------------- */

int findState(State *s)
{
    for (int i = 0; i < stateCount; i++)
    {
        if (sameState(&states[i], s))
            return i;
    }

    return -1;
}

/* -------------------------------------------------- */
/* Display item                                        */
/* -------------------------------------------------- */

void printItem(Item item)
{
    int p = item.production;
    int dot = item.dot;

    printf("%c -> ", prod[p].lhs);

    for (int i = 0; i < strlen(prod[p].rhs); i++)
    {
        if (i == dot)
            printf(".");

        printf("%c", prod[p].rhs[i]);
    }

    if (dot == strlen(prod[p].rhs))
        printf(".");

    printf("\n");
}

/* -------------------------------------------------- */
/* Display all LR(0) states                            */
/* -------------------------------------------------- */

void printStates()
{
    printf("\n============================================\n");
    printf("          LR(0) ITEM SETS\n");
    printf("============================================\n");

    for (int i = 0; i < stateCount; i++)
    {
        printf("\nI%d:\n", i);

        for (int j = 0; j < states[i].count; j++)
        {
            printf("  ");
            printItem(states[i].items[j]);
        }
    }
}

/* -------------------------------------------------- */
/* Main                                                */
/* -------------------------------------------------- */

int main()
{
    char input[MAX_LEN];

    printf("============================================\n");
    printf("          LR(0) PARSER\n");
    printf("============================================\n");

    printf("\nEnter number of productions: ");
    scanf("%d", &n);

    if (n <= 0 || n >= MAX_PROD)
    {
        printf("Invalid number of productions.\n");
        return 1;
    }

    printf("\nEnter productions without spaces.\n");
    printf("Example: E->E+T\n");
    printf("Use uppercase letters for non-terminals.\n\n");

    /*
       Production 0 is the original start production.
       We automatically create an augmented production
       S' -> S.
    */

    printf("Production 1: ");
    scanf("%c", &input[0]);
    scanf("%49[^\n]", input);

    /*
       Read all user productions first.
    */

    char original[MAX_PROD][MAX_LEN];

    strcpy(original[0], input);

    for (int i = 1; i < n; i++)
    {
        printf("Production %d: ", i + 1);
        scanf("%49s", original[i]);
    }

    /*
       Shift productions to make room for
       augmented production.
    */

    for (int i = n; i > 0; i--)
        strcpy(original[i], original[i - 1]);

    n++;

    /* Augmented production */
    prod[0].lhs = 'Z';
    strcpy(prod[0].rhs, original[1] + 3);

    /*
       Store original productions.
    */

    for (int i = 1; i < n; i++)
    {
        prod[i].lhs = original[i][0];
        strcpy(prod[i].rhs, original[i] + 3);
    }

    /*
       Create initial state.
    */

    states[0].count = 0;
    addItem(&states[0], 0, 0);
    closure(&states[0]);

    stateCount = 1;

    /*
       Generate canonical LR(0) collection.
    */

    int changed = 1;

    while (changed)
    {
        changed = 0;

        for (int i = 0; i < stateCount; i++)
        {
            char symbols[100];
            int symbolCount = 0;

            /*
               Find all symbols appearing after dots.
            */

            for (int j = 0; j < states[i].count; j++)
            {
                int p = states[i].items[j].production;
                int dot = states[i].items[j].dot;

                if (dot < strlen(prod[p].rhs))
                {
                    char c = prod[p].rhs[dot];

                    int exists = 0;

                    for (int k = 0; k < symbolCount; k++)
                    {
                        if (symbols[k] == c)
                        {
                            exists = 1;
                            break;
                        }
                    }

                    if (!exists)
                        symbols[symbolCount++] = c;
                }
            }

            /*
               Generate GOTO for every symbol.
            */

            for (int j = 0; j < symbolCount; j++)
            {
                State newState;

                goTo(&states[i],
                     symbols[j],
                     &newState);

                if (newState.count == 0)
                    continue;

                if (findState(&newState) == -1)
                {
                    states[stateCount] = newState;
                    stateCount++;
                    changed = 1;
                }
            }
        }
    }

    /* Display LR(0) states */

    printStates();

    /*
       Input string for parsing
    */

    printf("\nEnter input string to parse: ");
    scanf("%s", input);

    /*
       Add end marker
    */

    strcat(input, "$");

    printf("\n============================================\n");
    printf("Input String: %s\n", input);
    printf("============================================\n");

    /*
       Simple LR(0) recognition.
       The state transitions are calculated
       dynamically from the LR(0) item sets.
    */

    int stateStack[MAX_STACK];
    char symbolStack[MAX_STACK];

    int stateTop = 0;
    int symbolTop = -1;

    stateStack[0] = 0;

    int ip = 0;

    while (1)
    {
        int currentState = stateStack[stateTop];
        char currentSymbol = input[ip];

        int actionFound = 0;

        /*
           SHIFT
        */

        for (int i = 0; i < states[currentState].count; i++)
        {
            int p = states[currentState].items[i].production;
            int dot = states[currentState].items[i].dot;

            if (dot < strlen(prod[p].rhs) &&
                prod[p].rhs[dot] == currentSymbol)
            {
                State nextState;

                goTo(&states[currentState],
                     currentSymbol,
                     &nextState);

                int next = findState(&nextState);

                if (next != -1)
                {
                    printf("\nAction: SHIFT '%c'\n",
                           currentSymbol);

                    symbolStack[++symbolTop] =
                        currentSymbol;

                    stateStack[++stateTop] = next;

                    ip++;

                    actionFound = 1;
                    break;
                }
            }
        }

        if (actionFound)
            continue;

        /*
           REDUCE
        */

        for (int i = 0; i < states[currentState].count; i++)
        {
            int p = states[currentState].items[i].production;
            int dot = states[currentState].items[i].dot;

            /*
               Completed production
            */

            if (dot == strlen(prod[p].rhs))
            {
                /*
                   Augmented production Z -> S
                */

                if (p == 0 && currentSymbol == '$')
                {
                    printf("\nAction: ACCEPT\n");

                    printf("\n============================================\n");
                    printf("Input string ACCEPTED.\n");
                    printf("============================================\n");

                    return 0;
                }

                int rhsLen = strlen(prod[p].rhs);

                printf("\nAction: REDUCE by %c -> %s\n",
                       prod[p].lhs,
                       prod[p].rhs);

                /*
                   Pop RHS symbols and states
                */

                for (int k = 0; k < rhsLen; k++)
                {
                    if (symbolTop >= 0)
                        symbolTop--;

                    if (stateTop > 0)
                        stateTop--;
                }

                /*
                   Push LHS
                */

                symbolStack[++symbolTop] =
                    prod[p].lhs;

                /*
                   Find GOTO state
                */

                int prevState =
                    stateStack[stateTop];

                State nextState;

                goTo(&states[prevState],
                     prod[p].lhs,
                     &nextState);

                int next = findState(&nextState);

                if (next == -1)
                {
                    printf("\nParser Error.\n");
                    return 0;
                }

                stateStack[++stateTop] = next;

                actionFound = 1;
                break;
            }
        }

        if (actionFound)
            continue;

        /*
           ERROR
        */

        printf("\nAction: ERROR\n");

        printf("\n============================================\n");
        printf("Input string REJECTED.\n");
        printf("============================================\n");

        return 0;
    }

    return 0;
}

Writing lr_parser.c


In [ ]:
%%shell
gcc lr_parser.c -o lr_parser
./lr_parser

          LR(0) PARSER

Enter number of productions: 3

Enter productions without spaces.
Example: E->E+T
Use uppercase letters for non-terminals.

Production 1: S->CC
Production 2: C->cC
Production 3:  C->d

          LR(0) ITEM SETS

I0:
  Z -> .CC
  C -> .cC
  C -> .d

I1:
  Z -> C.C
  C -> .cC
  C -> .d

I2:
  C -> c.C
  C -> .cC
  C -> .d

I3:
  C -> d.

I4:
  Z -> CC.

I5:
  C -> cC.

Enter input string to parse: ccd

Input String: ccd$

Action: SHIFT 'c'

Action: SHIFT 'c'

Action: SHIFT 'd'

Action: REDUCE by C -> d

Action: REDUCE by C -> cC

Action: REDUCE by C -> cC

Action: ERROR

Input string REJECTED.
